##### Camerano Alessio - Matricola 1044444
##### Corso di Elaborazione di Immagini e Visione Artificiale
---
# Riconoscimento di Sagome Ossee tramite R-Table e Generalized Hough
---

### Cella 1: Costruzione fedele della R-table
Questa cella implementa passo per passo l'algoritmo di costruzione della R-table descritto da Ballard.
- Si sceglie un unico riferimento di forma \(y\) sul template (in questa implementazione: centroide della sagoma binaria).
- Per ogni punto di bordo \(x\), si calcola la direzione del gradiente \(\phi(x)\).
- Si costruisce il vettore \(r = y - x\).
- Si memorizza \(r\) come funzione di \(\phi\), ottenendo una mappa multi-valore \(R(\phi)\).

In [3]:
import time
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def list_image_files(folder):
    png_files = sorted(folder.glob('*.[pP][nN][gG]'))
    if png_files:
        return png_files
    return png_files


def load_gray_image(file_path):
    # Ballard lavora sul contorno: il colore non serve per la R-table.
    with Image.open(file_path) as img:
        return np.array(img.convert('L'))


def binarize_image(img_gray):
    # Otsu rende la segmentazione più stabile senza scegliere a mano la soglia.
    _, img_bin = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return img_bin


def edge_map_and_orientation(img_bin, threshold):
    # La direzione del gradiente è l'indice che Ballard usa per riempire la R-table.
    gx = cv2.Sobel(img_bin, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_bin, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.hypot(gx, gy)
    orientation = (np.degrees(np.arctan2(gy, gx)) % 360).astype(np.int32)
    edge_mask = magnitude > threshold
    ys, xs = np.where(edge_mask)
    return ys, xs, orientation, magnitude


def build_r_table(img_bin, threshold):
    # Scelgo il baricentro come riferimento y
    moments = cv2.moments(img_bin)
    if moments['m00'] == 0:
        raise ValueError('Il template non contiene pixel validi.')

    cx_ref = moments['m10'] / moments['m00']
    cy_ref = moments['m01'] / moments['m00']

    ys, xs, orientation, _ = edge_map_and_orientation(img_bin, threshold)
    r_table = {}

    for y, x in zip(ys, xs):
        # Ogni bordo memorizza il vettore r = y - x, esattamente come nella R-table di Ballard.
        phi = int(orientation[y, x])
        r_table.setdefault(phi, []).append((cy_ref - y, cx_ref - x))

    r_table = {phi: np.asarray(v, dtype=np.float32) for phi, v in sorted(r_table.items())}
    return r_table, (cx_ref, cy_ref), (ys, xs), orientation


path_template = Path('template')
template_files = list_image_files(path_template)
if not template_files:
    raise ValueError("La cartella 'template' è vuota.")

# Uso il primo template come forma di riferimento, nel flusso classico di Ballard.
reference_template_path = template_files[0]
SOGLIA_BORDO = 100

print('--- FASE 1: COSTRUZIONE DELL\'R-TABLE SECONDO BALLARD ---')
print(f"Template di riferimento: {reference_template_path.name}")

template_gray = load_gray_image(reference_template_path)
template_bin = binarize_image(template_gray)
r_table_base, centroide_riferimento, punti_bordo_template, orientamenti_template = build_r_table(
    template_bin,
    SOGLIA_BORDO,
)

num_vettori_template = sum(v.shape[0] for v in r_table_base.values())
print(f"Centroide di riferimento: ({centroide_riferimento[0]:.2f}, {centroide_riferimento[1]:.2f})")
print(f"Vettori complessivi nella R-table: {num_vettori_template}")


--- FASE 1: COSTRUZIONE DELL'R-TABLE SECONDO BALLARD ---
Template di riferimento: Bone-1.png
Centroide di riferimento: (176.74, 222.53)
Vettori complessivi nella R-table: 3420


### Cella 2: Generalized Hough 
Questa cella segue la procedura di Ballard per il riconoscimento di una singola forma tramite R-table.
- Per ogni pixel di bordo scena \(x\), si incrementano tutti i punti \(x+r\) dell'accumulatore, con \(r \in R(\phi)\).
- L'accumulatore base è su \(y\) (coordinate del riferimento), poi viene esteso a \(A(y,s,\theta)\).
- Le trasformazioni sono applicate come nel paper:
  - scala: \(T_s[R(\phi)] = sR(\phi)\),
  - rotazione: lookup con \((\phi-\theta)\) e rotazione dei vettori \(r\).


In [ ]:
path_scene = Path('test_diagnostics')
scene_files = list_image_files(path_scene)

if not scene_files:
    raise ValueError("La cartella è vuota.")

print('--- FASE 2: CARICAMENTO SCENE ---')
immagini_binarizzate = {}
for scene_path in scene_files:
    scene_gray = load_gray_image(scene_path)
    immagini_binarizzate[scene_path.name] = binarize_image(scene_gray)
print(f"Scene caricate: {len(immagini_binarizzate)}")


scale_test = [round(x, 2) for x in np.geomspace(0.25, 4.0, 11)]
ANGLE_STEP_DEG = 15  # passo più fine per recuperare rotazioni moderate senza esplodere i costi
rotazioni_test = list(range(0, 360, ANGLE_STEP_DEG))

soglia_confidenza = 0.05


def precompute_transformed_tables(r_table, scales, angles):
    transformed = {}
    for s in scales:
        for theta in angles:
            theta_rad = np.radians(theta)
            cos_t, sin_t = np.cos(theta_rad), np.sin(theta_rad)
            key = (s, theta)
            transformed[key] = {}
            
            for phi_obs in range(360):
                # Allineamento del lookup alla rotazione ipotizzata
                phi_idx = (phi_obs - theta) % 360
                vecs = r_table.get(phi_idx)
                
                if vecs is None:
                    continue
                
                dr = vecs[:, 0] # Componente Y (righe)
                dc = vecs[:, 1] # Componente X (colonne)
                
                # Applicazione corretta della matrice di rotazione per coordinate d'immagine (Y, X)
                # dove Y va verso il basso e X verso destra
                dr_t = s * (dr * cos_t + dc * sin_t)
                dc_t = s * (-dr * sin_t + dc * cos_t)
                
                transformed[key][phi_obs] = np.stack((dr_t, dc_t), axis=1).astype(np.float32)
    return transformed



transformed_tables = precompute_transformed_tables(
    r_table_base,
    scale_test,
    rotazioni_test,
)


print('--- FASE 3: VOTAZIONE GENERALIZED HOUGH ---')
start_total = time.time()
report = []

for nome_scena, img_scena_bin in immagini_binarizzate.items():
    start_scene = time.time()
    h, w = img_scena_bin.shape

    ys, xs, orientazioni_scena, _ = edge_map_and_orientation(img_scena_bin, SOGLIA_BORDO)
    if ys.size == 0:
        report.append((nome_scena, False, 0.0, None, None, None, time.time() - start_scene))
        continue

    mappa_pixel_scena = {}
    phi_scene = orientazioni_scena[ys, xs]
    for phi in range(360):
        mask = phi_scene == phi
        if np.any(mask):
            mappa_pixel_scena[phi] = (ys[mask], xs[mask])

    accumulatore_3d = np.zeros((h, w, len(scale_test), len(rotazioni_test)), dtype=np.float32)

    for idx_s, s in enumerate(scale_test):
        for idx_t, theta in enumerate(rotazioni_test):
            table_cfg = transformed_tables[(s, theta)]
            acc_plane = np.zeros((h, w), dtype=np.float32)

            for phi_obs, (y_sub, x_sub) in mappa_pixel_scena.items():
                # Ogni bordo scena vota tutte le ipotesi compatibili con la sua orientazione.
                vecs = table_cfg.get(phi_obs)
                if vecs is None:
                    continue

                centri_y = y_sub[:, None] + vecs[:, 0][None, :]
                centri_x = x_sub[:, None] + vecs[:, 1][None, :]

                centri_y_int = np.rint(centri_y).astype(np.int32)
                centri_x_int = np.rint(centri_x).astype(np.int32)

                validi = (
                    (centri_y_int >= 0)
                    & (centri_y_int < h)
                    & (centri_x_int >= 0)
                    & (centri_x_int < w)
                )

                if np.any(validi):
                    # Il voto finisce nel centro candidato y: è il cuore dell'accumulatore di Ballard.
                    np.add.at(
                        acc_plane,
                        (centri_y_int[validi], centri_x_int[validi]),
                        1.0,
                    )

            # Applica filtro gaussiano per smussare i picchi dell'accumulatore.
            accumulatore_3d[:, :, idx_s, idx_t] = cv2.GaussianBlur(acc_plane, (7, 7), 0)

    max_value = float(accumulatore_3d.max())
    y_star, x_star, s_star_idx, t_star_idx = np.unravel_index(np.argmax(accumulatore_3d), accumulatore_3d.shape)

    # La soglia finale è empirica: confronta il picco con la dimensione della R-table.
    confidenza = max_value / max(1, num_vettori_template)
    trovato = confidenza >= soglia_confidenza

    tempo = time.time() - start_scene
    report.append(
        (
            nome_scena,
            trovato,
            confidenza,
            (int(y_star), int(x_star)),
            scale_test[s_star_idx],
            rotazioni_test[t_star_idx],
            tempo,
        )
    )


positivi = [r for r in report if r[1]]
negativi = [r for r in report if not r[1]]

for nome_scena, trovato, conf, centro, scala, rot, tempo in report:
    if trovato:
        print(
            f"[TROVATO] {nome_scena:>15s} | conf={conf:6.2%} | y*={centro[0]:3d}, x*={centro[1]:3d} | s={scala:3.2f} | theta={rot:3d} | {tempo:5.2f}s"
        )
    else:
        print(f"[NO]      {nome_scena:>15s} | conf={conf:6.2%} | {tempo:5.2f}s")

print('\n==================================================')
print(f"Template riferimento: {reference_template_path.name}")
print(f"Immagini analizzate: {len(report)}")
print(f"Positivi (soglia sperimentale {soglia_confidenza:.0%}): {len(positivi)}")
print(f"Negativi: {len(negativi)}")
print(f"Tempo totale: {time.time() - start_total:.2f}s")
print('==================================================')


--- FASE 2: CARICAMENTO SCENE ---
Scene caricate: 8
--- FASE 3: VOTAZIONE GENERALIZED HOUGH ---
[TROVATO]      Bone-1.png | conf=25.97% | y*=223, x*=177 | s=1.00 | theta=  0 |  2.49s
[NO]           Bone-2.png | conf= 3.80% |  2.49s
[NO]           Bone-3.png | conf= 4.38% |  2.85s
[TROVATO] Bone-4-rescaled.png | conf=22.79% | y*= 55, x*= 44 | s=0.25 | theta=  0 |  0.20s
[TROVATO]      Bone-4.png | conf=20.57% | y*=285, x*=282 | s=1.00 | theta=315 |  2.83s
[TROVATO]      Bone-5.png | conf=25.86% | y*=180, x*=223 | s=1.00 | theta=270 |  1.90s
[NO]           Bone-6.png | conf= 3.83% |  2.71s
[NO]            bat-4.png | conf= 4.57% |  0.43s

Template riferimento: Bone-1.png
Immagini analizzate: 8
Positivi (soglia sperimentale 5%): 4
Negativi: 4
Tempo totale: 15.91s


# Conclusioni

## Note pratiche

- **SIFT:** utile come confronto moderno, ma non fa parte del metodo Ballard.
- **Canny:** può pulire i bordi, ma cambia il preprocessing, non il cuore dell'algoritmo.
- **Deep Learning:** è un'alternativa diversa, non una variante di Ballard.

## Quando usare il metodo

- **Adatto a:** sagome rigide, contorni chiari, variazioni di posa e di scala entro un insieme di ipotesi discretizzato.
- **Limiti:** sensibilità alla quantizzazione di scala/angolo, costo computazionale crescente e scarsa affidabilità sui ribaltamenti di forme asimmetriche.